# 📖 Notebook 2: Real-Time Aggregation with Sliding Windows

In Notebook 1 we ingested click events into Kafka and stored them in PostgreSQL. But querying raw events is **too slow** for advertisers who need instant dashboards.

The solution? **Pre-aggregate** clicks into 1-minute buckets *as they stream in*. When an advertiser asks "how many clicks did my Nike ad get in the last hour?", we just SUM 60 pre-computed rows instead of scanning millions of raw events.

```
Raw events (millions)          Pre-aggregated (one row per ad per minute)
┌────────────────────┐         ┌──────────────────────────────────┐
│ ad=1, 10:00:03     │         │ ad=1  | 10:00 | 47 clicks       │
│ ad=1, 10:00:15     │  ───▶   │ ad=1  | 10:01 | 52 clicks       │
│ ad=1, 10:00:47     │         │ ad=2  | 10:00 | 31 clicks       │
│ ad=2, 10:00:02     │         │ ...                              │
│ ...millions more   │         └──────────────────────────────────┘
└────────────────────┘         Much faster to query!
```

## Learning Objectives

By the end of this notebook you will understand:
- The difference between **event time** and **processing time**
- How **tumbling windows** (fixed 1-minute buckets) work
- How to build an aggregation consumer that writes to `click_aggregates`
- How **late-arriving events** can land in the wrong window, and how a **watermark**
  decides which of them still get counted and which get dropped
- Why **at-least-once** delivery double-counts, and how an **idempotency key** fixes it
- How **sliding windows** compose out of tumbling ones — and the gap trap that breaks them
- How a batch **reconciliation** job repairs the streaming numbers
- Why pre-aggregation makes advertiser queries sub-second

## 🛠️ Setup

Make sure the infrastructure is running:

```bash
cd 06-system-designs/ad-click-aggregator
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL): http://localhost:8080
- **Kafka UI**: http://localhost:8081

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import json
import time
import uuid
import random
from datetime import datetime, timezone, timedelta
from collections import defaultdict, Counter
from confluent_kafka import Producer, Consumer, KafkaError
from confluent_kafka.admin import AdminClient, NewTopic

# ── Connection settings ─────────────────────────────────────
DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "adclick_demo", "user": "demo", "password": "demo"
}
KAFKA_BROKER = "localhost:9094"

# ── Determinism ─────────────────────────────────────────────
# Seeded so the counts in the prose are the counts you get. RUN_ID tags this
# run's events: Kafka topics outlive the notebook, so without it a second run
# would aggregate the first run's events and every "expected count" below
# would be a lie.
random.seed(42)
RUN_ID = uuid.uuid4().hex[:8]
print(f"🔖 run id: {RUN_ID}")

def get_db():
    return psycopg2.connect(**DB_CONFIG)

# Verify connections
try:
    conn = get_db(); conn.close()
    print("✅ PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL: {e}")

try:
    admin = AdminClient({"bootstrap.servers": KAFKA_BROKER})
    admin.list_topics(timeout=5)
    print("✅ Kafka")
except Exception as e:
    print(f"❌ Kafka: {e}")

## ⏰ Event Time vs Processing Time

This is one of the most important concepts in stream processing. Let's understand the difference:

| Concept | Definition | Example |
|---------|------------|----------|
| **Event time** | When the click *actually happened* on the user's device | `10:00:03 UTC` |
| **Processing time** | When our server *received and processed* the event | `10:00:05 UTC` |

Why the difference? Network delays, server load, or the user's phone being briefly offline can all cause events to arrive late.

**We always aggregate by event time** because that's when the click truly occurred. Using processing time would produce inaccurate counts.

In [ ]:
# Let's demonstrate the problem with a concrete example.
# Nothing below is hard-coded: we bucket the SAME five clicks twice — once by
# event time, once by processing time — and let the counts speak.

def minute_bucket(hhmmss: str) -> str:
    """'10:00:58' -> '10:00'. A 1-minute tumbling window, half-open: [10:00, 10:01)."""
    return hhmmss[:5]

print("⏰ Event Time vs Processing Time")
print("=" * 60)
print()

# Simulate 5 clicks that happened at known event times
# but arrived at our server in a different order
clicks = [
    {"event_time": "10:00:03", "processed_at": "10:00:04", "note": "arrived on time"},
    {"event_time": "10:00:15", "processed_at": "10:00:16", "note": "arrived on time"},
    {"event_time": "10:00:45", "processed_at": "10:00:46", "note": "arrived on time"},
    {"event_time": "10:00:58", "processed_at": "10:01:03", "note": "⚠️ arrived LATE (5s delay)"},
    {"event_time": "10:01:02", "processed_at": "10:01:02", "note": "arrived on time"},
]

print(f"{'Event Time':<14} {'Processed At':<14} {'Note'}")
print("-" * 60)
for c in clicks:
    print(f"{c['event_time']:<14} {c['processed_at']:<14} {c['note']}")

by_event = Counter(minute_bucket(c["event_time"]) for c in clicks)
by_processing = Counter(minute_bucket(c["processed_at"]) for c in clicks)

print()
print("📊 If we aggregate by EVENT TIME (correct):")
for window in sorted(by_event):
    print(f"   Window {window} → {by_event[window]} clicks")
print()
print("📊 If we aggregate by PROCESSING TIME (wrong):")
for window in sorted(by_processing):
    print(f"   Window {window} → {by_processing[window]} clicks")

misplaced = [c for c in clicks
             if minute_bucket(c["event_time"]) != minute_bucket(c["processed_at"])]
print()
print(f"⚠️  {len(misplaced)} click(s) land in the wrong window under processing time:")
for c in misplaced:
    print(f"   clicked {c['event_time']}, processed {c['processed_at']} → counted in "
          f"{minute_bucket(c['processed_at'])} instead of {minute_bucket(c['event_time'])}")
print("   Always use EVENT TIME for accurate aggregation.")

assert by_event == {"10:00": 4, "10:01": 1}, f"event-time buckets drifted: {dict(by_event)}"
assert by_processing == {"10:00": 3, "10:01": 2}, \
    f"processing-time buckets drifted: {dict(by_processing)}"
assert len(misplaced) == 1, "the demo needs exactly one click that crosses a boundary"


## 🪟 Tumbling Windows

A **tumbling window** is a fixed-size, non-overlapping time bucket. For our system, each window is exactly 1 minute long:

```
Time ──────────────────────────────────────────────▶

│  Window 1   │  Window 2   │  Window 3   │
│ 10:00-10:01 │ 10:01-10:02 │ 10:02-10:03 │
│  ● ● ●  ●  │  ●  ●       │  ● ● ● ● ● │
│  4 clicks   │  2 clicks   │  5 clicks   │
```

Every click goes into exactly **one** window based on its event time. No overlaps, no gaps.

Let's build a function that calculates which window a click belongs to.

In [ ]:
def get_window_start(event_time_str: str, window_seconds: int = 60) -> datetime:
    """
    Given an event time, return the start of its tumbling window.

    Windows are HALF-OPEN: [start, start + window_seconds). An event landing
    exactly on a boundary belongs to the window that OPENS there, never to the
    one that closes there — which is what guarantees every click lands in
    exactly one window, with no double counting and no gaps.

    Example: event at 10:00:43 with 60s windows → window starts at 10:00:00
    """
    event_time = datetime.fromisoformat(event_time_str)
    if event_time.tzinfo is None:
        # A timestamp with no offset would otherwise be read as the reader's
        # LOCAL time and land hours away from its real window. Clicks are UTC.
        event_time = event_time.replace(tzinfo=timezone.utc)
    # Truncate to the start of the window
    timestamp = event_time.timestamp()
    window_start_ts = (timestamp // window_seconds) * window_seconds
    return datetime.fromtimestamp(window_start_ts, tz=timezone.utc)

# Demonstrate with some example times
examples = [
    "2026-03-31T10:00:03+00:00",
    "2026-03-31T10:00:45+00:00",
    "2026-03-31T10:00:59+00:00",
    "2026-03-31T10:01:00+00:00",
    "2026-03-31T10:01:30+00:00",
]

print("🪟 Tumbling Window Assignment (1-minute windows)")
print("=" * 55)
print(f"{'Event Time':<30} {'Window Start':<20}")
print("-" * 55)
for et in examples:
    ws = get_window_start(et)
    short_event = et[11:19]
    short_window = ws.strftime("%H:%M:%S")
    print(f"  {short_event:<28} {short_window:<20}")

print()
print("💡 10:00:59 and 10:01:00 are just 1 second apart")
print("   but land in DIFFERENT windows. That's tumbling windows!")


# ── Assertions: window boundaries are where counting silently goes wrong ────
def utc(hour, minute):
    return datetime(2026, 3, 31, hour, minute, tzinfo=timezone.utc)

# One second either side of a boundary must split.
assert get_window_start("2026-03-31T10:00:59+00:00") == utc(10, 0)
assert get_window_start("2026-03-31T10:01:00+00:00") == utc(10, 1)
# The boundary instant itself belongs to the window it OPENS (half-open).
assert get_window_start("2026-03-31T10:01:00+00:00") != utc(10, 0)
# First and last instant of a window both resolve to that window.
assert get_window_start("2026-03-31T10:01:00.000001+00:00") == utc(10, 1)
assert get_window_start("2026-03-31T10:01:59.999999+00:00") == utc(10, 1)
# A timestamp without an offset is read as UTC, not as the reader's local time.
assert get_window_start("2026-03-31T10:01:00") == utc(10, 1)
# Non-default window sizes floor the same way.
assert get_window_start("2026-03-31T10:04:59+00:00", 300) == utc(10, 0)
assert get_window_start("2026-03-31T10:05:00+00:00", 300) == utc(10, 5)
print("✅ boundary assertions pass — every click lands in exactly one window")

## 🚀 Step 1 — Produce Click Events with Controlled Timestamps

Let's produce events with specific timestamps so we can verify our aggregation is correct.

In [ ]:
topic_name = "ad-clicks-agg"

# Create topic if needed
admin = AdminClient({"bootstrap.servers": KAFKA_BROKER})
existing = admin.list_topics(timeout=10).topics
if topic_name not in existing:
    futures = admin.create_topics([NewTopic(topic_name, num_partitions=3, replication_factor=1)])
    for t, f in futures.items():
        f.result()
    print(f"✅ Created topic '{topic_name}'")
else:
    print(f"ℹ️  Topic '{topic_name}' already exists")

# Produce events across 3 one-minute windows
producer = Producer({"bootstrap.servers": KAFKA_BROKER})

base_time = datetime(2026, 3, 31, 10, 0, 0, tzinfo=timezone.utc)

# We'll create a known distribution so we can verify our aggregation
# Window 10:00 → ad 1 gets 5 clicks, ad 2 gets 3 clicks
# Window 10:01 → ad 1 gets 2 clicks, ad 2 gets 7 clicks
# Window 10:02 → ad 1 gets 4 clicks, ad 2 gets 1 click
planned_events = []
distributions = [
    (0, 1, 5),   # minute 0, ad 1, 5 clicks
    (0, 2, 3),   # minute 0, ad 2, 3 clicks
    (1, 1, 2),   # minute 1, ad 1, 2 clicks
    (1, 2, 7),   # minute 1, ad 2, 7 clicks
    (2, 1, 4),   # minute 2, ad 1, 4 clicks
    (2, 2, 1),   # minute 2, ad 2, 1 click
]

for minute_offset, ad_id, count in distributions:
    for i in range(count):
        event_time = base_time + timedelta(minutes=minute_offset, seconds=random.randint(0, 59))
        event = {
            "ad_id": ad_id,
            "impression_id": str(uuid.uuid4()),
            "user_id": f"user_{random.randint(1, 200)}",
            "ip_address": f"10.0.{random.randint(1,254)}.{random.randint(1,254)}",
            "user_agent": "Mozilla/5.0",
            "event_time": event_time.isoformat(),
            "run_id": RUN_ID,   # so a re-run ignores the events of earlier runs
        }
        planned_events.append(event)

# The expected result, derived from the same plan the events came from — so it
# can never drift out of sync with the prose the way a hand-typed table would.
expected_counts = {
    (ad_id, (base_time + timedelta(minutes=minute_offset)).isoformat()): count
    for minute_offset, ad_id, count in distributions
}

# Shuffle to simulate out-of-order arrival (realistic!)
random.shuffle(planned_events)

for event in planned_events:
    producer.produce(
        topic=topic_name,
        key=str(event["ad_id"]),
        value=json.dumps(event)
    )

producer.flush(timeout=10)

print(f"✅ Produced {len(planned_events)} events (shuffled to simulate out-of-order)")
print()
print("📊 Expected aggregation:")
for (ad_id, window_start), count in sorted(expected_counts.items(),
                                           key=lambda kv: (kv[0][1], kv[0][0])):
    print(f"   Window {window_start[11:16]} → ad {ad_id}: {count} clicks")

## 🔄 Step 2 — Build the Aggregation Consumer

This is the heart of the system. Our consumer will:

1. **Read** click events from Kafka
2. **Assign** each event to a 1-minute window using its **event time**
3. **Accumulate** counts in memory (a dictionary)
4. **Flush** the counts to the `click_aggregates` table

In production, Flink does this automatically with watermarks and exactly-once guarantees. Here, we'll build a simplified version to understand the mechanics.

In [ ]:
def aggregate_clicks(topic: str, max_events: int = 100, timeout_s: int = 15,
                     run_id: str | None = None):
    """
    Consume events from Kafka and aggregate into 1-minute windows.

    `run_id` filters out events left in the topic by earlier runs of this
    notebook — without it, a re-run silently aggregates stale data.

    Returns a dict: {(ad_id, window_start): {"click_count": N, "unique_users": set()}}
    """
    consumer = Consumer({
        "bootstrap.servers": KAFKA_BROKER,
        "group.id": "agg-consumer-" + str(uuid.uuid4())[:8],
        "auto.offset.reset": "earliest",
        "enable.auto.commit": False
    })
    consumer.subscribe([topic])

    # In-memory aggregation state
    # Key: (ad_id, window_start_str) → Value: {click_count, unique_users}
    windows = defaultdict(lambda: {"click_count": 0, "unique_users": set()})

    consumed = 0
    start = time.time()

    while time.time() - start < timeout_s and consumed < max_events:
        msg = consumer.poll(timeout=1.0)
        if msg is None:
            continue
        if msg.error():
            if msg.error().code() == KafkaError._PARTITION_EOF:
                continue
            break

        event = json.loads(msg.value().decode("utf-8"))
        if run_id is not None and event.get("run_id") != run_id:
            continue  # produced by an earlier run of this notebook — not ours
        ad_id = event["ad_id"]
        event_time = event["event_time"]
        user_id = event.get("user_id", "anonymous")

        # Assign to window using EVENT TIME (not processing time!)
        window_start = get_window_start(event_time)
        key = (ad_id, window_start.isoformat())

        windows[key]["click_count"] += 1
        windows[key]["unique_users"].add(user_id)

        consumed += 1

    consumer.close()
    return consumed, dict(windows)

# Run the aggregation
consumed, windows = aggregate_clicks(topic_name, max_events=len(planned_events),
                                     run_id=RUN_ID)

print(f"📥 Consumed {consumed} events")
print(f"📊 Aggregated into {len(windows)} windows:\n")

print(f"{'Ad ID':<8} {'Window Start':<22} {'Clicks':>8} {'Unique Users':>13}")
print("-" * 55)
for (ad_id, window_start), data in sorted(windows.items()):
    short_window = window_start[11:16]  # just HH:MM
    print(f"{ad_id:<8} {short_window:<22} {data['click_count']:>8} {len(data['unique_users']):>13}")

# ── Assertion: the aggregation has to reproduce the plan, exactly ──────────
actual_counts = {key: data["click_count"] for key, data in windows.items()}
assert actual_counts == expected_counts, (
    "windowed counts do not match the plan that produced the events\n"
    f"  expected: {sorted(expected_counts.items())}\n"
    f"  actual:   {sorted(actual_counts.items())}"
)
assert consumed == len(planned_events), \
    f"expected {len(planned_events)} events from run {RUN_ID}, consumed {consumed}"

print("\n✅ Every window matches the planned counts exactly — including the events")
print("   that arrived out of order. Event-time windowing does not care about")
print("   arrival order; it only cares about the timestamp inside the event.")

## 💾 Step 3 — Flush Aggregates to PostgreSQL

Now let's write the aggregated data to the `click_aggregates` table. We use **UPSERT** (`ON CONFLICT ... DO UPDATE`) so that if more events arrive for the same window, we just add to the existing count.

This is exactly what Flink does when it flushes its window state to the database.

> ⚠️ **`unique_users` is not actually unique.** Adding two flushes' unique-user
> counts double-counts anybody who appears in both — the count is an upper
> bound, not a distinct count. `click_count` is exact; `unique_users` is not.
> Real systems store a mergeable sketch (HyperLogLog) per window and merge the
> sketches instead of adding integers. We keep the integer here because the
> point of this notebook is windowing, not cardinality estimation — but do not
> read that column as truth.

> ⚠️ **This UPSERT is additive, which means it is *not* safe to apply twice.**
> That is a real bug waiting to happen, and Step 6 reproduces it before fixing it.

In [ ]:
def flush_to_postgres(windows: dict):
    """
    Write aggregated window data to the click_aggregates table.
    Uses UPSERT so repeated flushes accumulate correctly.
    """
    conn = get_db()
    cur = conn.cursor()

    upsert_sql = """
        INSERT INTO click_aggregates (ad_id, window_start, click_count, unique_users)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (ad_id, window_start)
        DO UPDATE SET
            click_count = click_aggregates.click_count + EXCLUDED.click_count,
            unique_users = click_aggregates.unique_users + EXCLUDED.unique_users,
            updated_at = CURRENT_TIMESTAMP
    """

    flushed = 0
    for (ad_id, window_start), data in windows.items():
        cur.execute(upsert_sql, (
            ad_id,
            window_start,
            data["click_count"],
            len(data["unique_users"])
        ))
        flushed += 1

    conn.commit()
    conn.close()
    return flushed

flushed = flush_to_postgres(windows)
print(f"✅ Flushed {flushed} window aggregates to PostgreSQL")

## 📊 Step 4 — Advertiser Query: Fast!

Now let's see how fast the advertiser query is when it reads from the pre-aggregated table instead of scanning millions of raw events.

In [ ]:
conn = get_db()
cur = conn.cursor()

# Fast query: just SUM the pre-aggregated rows
start = time.time()
cur.execute("""
    SELECT a.title, ca.window_start, ca.click_count, ca.unique_users
    FROM click_aggregates ca
    JOIN ads a ON a.id = ca.ad_id
    ORDER BY ca.window_start, a.title
""")
rows = cur.fetchall()
elapsed = (time.time() - start) * 1000
conn.close()

print(f"⚡ Query completed in {elapsed:.1f} ms\n")
print(f"{'Ad Title':<42} {'Window':<22} {'Clicks':>7} {'Unique':>7}")
print("-" * 82)
for row in rows:
    window_str = row[1].strftime("%Y-%m-%d %H:%M") if hasattr(row[1], 'strftime') else str(row[1])[:16]
    print(f"{row[0]:<42} {window_str:<22} {row[2]:>7} {row[3]:>7}")

print()
print("💡 This query touches only a few rows, not millions.")
print("   An advertiser asking 'how many clicks in the last hour?'")
print("   scans at most 60 rows per ad (one per minute) — instant!")

## ⚠️ Step 5 — Late-Arriving Events

What happens when a click event arrives *after* we've already closed and flushed its window?

For example, a user clicks an ad at 10:00:58 but due to network delays, the event doesn't reach our consumer until 10:01:05 — after the 10:00 window has been flushed.

### Strategies for Late Events

| Strategy | How It Works | Trade-off |
|----------|-------------|----------|
| **Drop** | Ignore events that arrive after the window closes | Simple but loses data |
| **Update** | Reopen the window and update the aggregate | Accurate but more complex |
| **Watermark** | Wait a grace period before closing the window | Good balance |

We implement **Update** and **Watermark** together, because they answer different questions:

- The **Update** strategy is what our UPSERT already gives us: as long as we are
  still willing to touch a window, a late event just adds to it.
- The **Watermark** is what decides *how long* we stay willing. It is the stream's
  own estimate of how far along in **event time** it has got — usually
  `max event time seen so far − allowed lateness`. Once the watermark passes the
  **end** of a window, that window is closed, and anything still arriving for it
  is too late to count.

Let's see the Update strategy first, then build a real watermark and watch a click
get dropped by it.

In [ ]:
# Simulate a late-arriving event
print("⚠️  Simulating a Late-Arriving Event")
print("=" * 50)

# Check current aggregate for ad 1 at 10:00
conn = get_db()
cur = conn.cursor()
cur.execute("""
    SELECT click_count FROM click_aggregates
    WHERE ad_id = 1 AND window_start = '2026-03-31 10:00:00+00'
""")
row = cur.fetchone()
before_count = row[0] if row else 0
print(f"\n📊 BEFORE: ad_id=1, window 10:00 has {before_count} clicks")

# A late event arrives for the 10:00 window
late_event_window = {
    (1, "2026-03-31T10:00:00+00:00"): {
        "click_count": 1,
        "unique_users": {"user_late_999"}
    }
}

print("\n⏰ Late event arrives: ad_id=1, event_time=10:00:58, arrived at 10:01:05")
print("   Our UPSERT adds it to the existing window...")

flush_to_postgres(late_event_window)

# Check after
cur.execute("""
    SELECT click_count FROM click_aggregates
    WHERE ad_id = 1 AND window_start = '2026-03-31 10:00:00+00'
""")
row = cur.fetchone()
after_count = row[0] if row else 0
conn.close()

print(f"\n📊 AFTER: ad_id=1, window 10:00 has {after_count} clicks")
print(f"   (+{after_count - before_count} from the late event)")

assert after_count == before_count + 1, (
    f"the late click must land in the 10:00 window: {before_count} -> {after_count}"
)

print("\n✅ The UPSERT handled the late event correctly!")
print("   No data was lost — the aggregate was updated in place.")
print("   But notice what this cell did NOT do: it never asked whether the 10:00")
print("   window was still open. Left like this, a click from three days ago would")
print("   rewrite a window an advertiser was already invoiced for. That is what a")
print("   watermark is for.")

### 🌊 A Real Watermark

A watermark is a single moving timestamp that answers *"how far along in event time
am I?"*. The usual rule:

```
watermark = (highest event time seen so far) − allowed_lateness
```

Two decisions fall out of it, and they are different:

| Question | Test | Outcome |
|---|---|---|
| Is this event late? | `event_time < watermark` | it is late, but we may still count it |
| Is its window closed? | `window_end <= watermark` | too late — the event is **dropped** |

Note the `<=` in the second test. Windows are half-open `[start, end)`, so a
watermark sitting exactly on `end` means every instant the window covers is already
in the past. The window closes.

Below, the same seven clicks arrive in an order that is *not* their event order.
Two of them will be dropped. Then we widen `allowed_lateness` and watch all seven
survive — the whole trade-off in two runs.

In [ ]:
ALLOWED_LATENESS_S = 30
WINDOW_S = 60


class WatermarkAggregator:
    """1-minute tumbling windows, a watermark, and a side output for stragglers."""

    def __init__(self, allowed_lateness_s: int = ALLOWED_LATENESS_S):
        self.allowed_lateness_s = allowed_lateness_s
        self.max_event_ts = None   # highest event time seen so far
        self.windows = defaultdict(lambda: {"click_count": 0, "unique_users": set()})
        self.dropped = []          # side output: clicks that missed their window

    @property
    def watermark(self):
        if self.max_event_ts is None:
            return None
        return self.max_event_ts - timedelta(seconds=self.allowed_lateness_s)

    def add(self, event: dict) -> str:
        """Returns 'on_time', 'late_merged' or 'dropped'."""
        event_ts = datetime.fromisoformat(event["event_time"])
        window_start = get_window_start(event["event_time"], WINDOW_S)
        window_end = window_start + timedelta(seconds=WINDOW_S)

        # Read the watermark BEFORE this event advances it. Otherwise an event
        # would be judged against a watermark it just moved, and nothing could
        # ever be late with respect to itself.
        wm = self.watermark

        if wm is not None and window_end <= wm:
            self.dropped.append(event)          # window already closed
            verdict = "dropped"
        else:
            key = (event["ad_id"], window_start.isoformat())
            self.windows[key]["click_count"] += 1
            self.windows[key]["unique_users"].add(event["user_id"])
            verdict = "on_time" if wm is None or event_ts >= wm else "late_merged"

        if self.max_event_ts is None or event_ts > self.max_event_ts:
            self.max_event_ts = event_ts
        return verdict


base = datetime(2026, 3, 31, 12, 0, 0, tzinfo=timezone.utc)


def click(second_offset: int, user: str) -> dict:
    return {"ad_id": 1, "user_id": user,
            "event_time": (base + timedelta(seconds=second_offset)).isoformat()}


# ARRIVAL order (processing time). The event times inside are deliberately jumbled.
arrivals = [
    click(5,   "u1"),   # 12:00:05
    click(20,  "u2"),   # 12:00:20
    click(58,  "u3"),   # 12:00:58
    click(70,  "u4"),   # 12:01:10 — pushes the watermark to 12:00:40
    click(35,  "u5"),   # 12:00:35 — behind the watermark, but 12:00 is still open
    click(150, "u6"),   # 12:02:30 — watermark jumps to 12:02:00, closing 12:00 AND 12:01
    click(30,  "u7"),   # 12:00:30 — window 12:00 is gone
    click(90,  "u8"),   # 12:01:30 — window 12:01 ended exactly ON the watermark
]

agg = WatermarkAggregator()

print("🌊 Watermark aggregation (allowed lateness = "
      f"{ALLOWED_LATENESS_S}s, windows = {WINDOW_S}s)")
print("=" * 78)
print(f"{'arrives':<9} {'event time':<12} {'window':<9} {'watermark':<12} {'verdict'}")
print("-" * 78)

verdicts = []
for n, event in enumerate(arrivals, 1):
    wm_before = agg.watermark
    verdict = agg.add(event)
    verdicts.append(verdict)
    window = get_window_start(event["event_time"], WINDOW_S).strftime("%H:%M")
    icon = {"on_time": "✅", "late_merged": "⏰", "dropped": "🗑️"}[verdict]
    print(f"#{n:<8} {event['event_time'][11:19]:<12} {window:<9} "
          f"{(wm_before.strftime('%H:%M:%S') if wm_before else '—'):<12} {icon} {verdict}")

print()
print("📊 Windows that survived:")
for (ad_id, window_start), data in sorted(agg.windows.items()):
    print(f"   ad {ad_id} | {window_start[11:16]} | {data['click_count']} clicks")

counted = sum(w["click_count"] for w in agg.windows.values())
print()
print(f"💧 {len(agg.dropped)} of {len(arrivals)} clicks fell past the watermark and were")
print("   DROPPED. Those are real clicks the advertiser will never be billed for —")
print("   silent, permanent under-counting, and the reason you keep the side output")
print("   instead of throwing it away.")

# ── The trade-off, run rather than described ───────────────────────────────
patient = WatermarkAggregator(allowed_lateness_s=180)
for event in arrivals:
    patient.add(event)
patient_counted = sum(w["click_count"] for w in patient.windows.values())

print()
print(f"⚖️  Same stream, allowed lateness 180s → {patient_counted}/{len(arrivals)} counted, "
      f"{len(patient.dropped)} dropped.")
print("   Waiting longer loses fewer clicks but holds window state in memory longer")
print("   and delays the final number. Waiting forever is not an option at 10k/s;")
print("   the batch reconciliation job in Step 9 is what recovers the rest.")

# ── Assertions ─────────────────────────────────────────────────────────────
assert verdicts.count("dropped") == 2, \
    f"the watermark must actually drop the two stragglers; verdicts={verdicts}"
assert verdicts.count("late_merged") == 1, \
    f"click #5 is late but its window is still open; verdicts={verdicts}"
assert agg.windows[(1, base.isoformat())]["click_count"] == 4, \
    "window 12:00 should hold u1, u2, u3 and the late-but-merged u5"
assert counted == len(arrivals) - 2, f"counted {counted}, expected {len(arrivals) - 2}"
# u8's window ends at 12:02:00 and the watermark is exactly 12:02:00 — half-open
# windows mean that instant is already past, so it closes. Off-by-one lives here.
assert agg.dropped[-1]["user_id"] == "u8", \
    "a window whose end equals the watermark is closed, not open"
assert not patient.dropped and patient_counted == len(arrivals), \
    "with 180s of patience nothing should be dropped"
print()
print("✅ watermark assertions pass")


## 💸 Step 6 — At-Least-Once Delivery Double-Counts

Our consumer does three things in a row: read a batch from Kafka, UPSERT the
aggregate, commit the offset. Those are **two different systems** and **two
separate writes**, with no transaction spanning them.

```
read batch  ──▶  UPSERT click_count += N   ✅ committed to Postgres
                        │
                        💥  process dies here
                        │
                 commit Kafka offset       ❌ never happened

restart  ──▶  Kafka replays the same batch  ──▶  click_count += N  again
```

Kafka gives you **at-least-once** delivery, and our UPSERT is **additive**, so a
replay adds the same clicks a second time. Nothing errors. Nothing looks broken.
The advertiser is simply billed twice.

This is the one defect in an ad system that ends in a refund and a lawyer, so we
are going to reproduce it before we fix it.

### The fix: an idempotency key

Give every flush a **deterministic id** derived from what it covers — the topic,
partition and offset range, not a random UUID — and record that id in the *same
transaction* that applies the counts, behind a primary key. A replay tries to
insert an id that already exists, the whole transaction is skipped, and the
numbers do not move.

That is the same shape as Flink's two-phase-commit sink and Kafka's transactional
producer: the offset and the output move together, or neither moves.

In [ ]:
# ── Reproduce the bug ──────────────────────────────────────────────────────
conn = get_db(); cur = conn.cursor()
cur.execute("DELETE FROM click_aggregates")
conn.commit(); conn.close()

CRASH_WINDOW = "2026-03-31T16:00:00+00:00"

# One batch of aggregated windows — pretend the consumer just built this from
# ten Kafka records it has not yet committed offsets for.
batch = {(1, CRASH_WINDOW): {"click_count": 10, "unique_users": {"alice", "bob"}}}


def count_at(ad_id: int, window_start: str) -> int:
    conn = get_db(); cur = conn.cursor()
    cur.execute(
        "SELECT click_count FROM click_aggregates WHERE ad_id = %s AND window_start = %s",
        (ad_id, window_start),
    )
    row = cur.fetchone()
    conn.close()
    return row[0] if row else 0


flush_to_postgres(batch)      # the flush succeeds...
#  💥 the consumer dies right here, before committing its Kafka offset
flush_to_postgres(batch)      # ...and the restart replays the very same batch

doubled = count_at(1, CRASH_WINDOW)

print("💸 At-Least-Once Double Count")
print("=" * 55)
print(f"   clicks actually in the batch : 10")
print(f"   click_count after one replay : {doubled}")
print()
print("   No exception. No warning. The advertiser is invoiced for "
      f"{doubled} clicks")
print("   when only 10 people clicked.")

assert doubled == 20, (
    f"the double-count must reproduce for the fix to mean anything; got {doubled}"
)


In [ ]:
# ── The fix: apply each batch exactly once ────────────────────────────────
conn = get_db(); cur = conn.cursor()
cur.execute("""
    CREATE TABLE IF NOT EXISTS applied_batches (
        batch_id    VARCHAR(128) PRIMARY KEY,
        applied_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")
cur.execute("DELETE FROM click_aggregates")
cur.execute("DELETE FROM applied_batches")
conn.commit(); conn.close()


def flush_idempotent(windows: dict, batch_id: str) -> bool:
    """Apply a batch of window counts at most once.

    Returns True if the batch was applied, False if this batch_id had already
    been applied — i.e. this call is a replay and was correctly ignored.

    The marker row and the counts go in ONE transaction. Either both land or
    neither does; there is no window in which the counts are applied but the
    system has forgotten that they were.
    """
    conn = get_db()
    try:
        with conn:  # psycopg2: commit on clean exit, rollback on exception
            cur = conn.cursor()
            cur.execute(
                "INSERT INTO applied_batches (batch_id) VALUES (%s) "
                "ON CONFLICT (batch_id) DO NOTHING",
                (batch_id,),
            )
            if cur.rowcount == 0:
                return False  # already applied — a replay, so do nothing
            for (ad_id, window_start), data in windows.items():
                cur.execute("""
                    INSERT INTO click_aggregates
                        (ad_id, window_start, click_count, unique_users)
                    VALUES (%s, %s, %s, %s)
                    ON CONFLICT (ad_id, window_start) DO UPDATE SET
                        click_count  = click_aggregates.click_count + EXCLUDED.click_count,
                        unique_users = click_aggregates.unique_users + EXCLUDED.unique_users,
                        updated_at   = CURRENT_TIMESTAMP
                """, (ad_id, window_start, data["click_count"], len(data["unique_users"])))
            return True
    finally:
        conn.close()


# Deterministic id: topic, partition, offset range. NOT a fresh uuid4 — a random
# id would be different on the replay and would defeat the whole mechanism.
batch_id = "ad-clicks-agg:p0:offsets-0-9"

first_apply = flush_idempotent(batch, batch_id)
#  💥 same crash, same replay
replay_apply = flush_idempotent(batch, batch_id)

final = count_at(1, CRASH_WINDOW)

print("🔑 Same crash, same replay, with an idempotency key")
print("=" * 55)
print(f"   first flush applied? {first_apply}")
print(f"   replay applied?      {replay_apply}   ← recognised and skipped")
print(f"   click_count          {final}")

assert first_apply is True, "the first flush must actually apply"
assert replay_apply is False, "the replay must be recognised as already applied"
assert final == 10, f"exactly-once flush must leave 10 clicks, got {final}"

# A *different* batch must still get through — an idempotency key that blocks
# everything is not idempotent, it is broken.
other = {(1, CRASH_WINDOW): {"click_count": 4, "unique_users": {"carol"}}}
assert flush_idempotent(other, "ad-clicks-agg:p0:offsets-10-13") is True
assert count_at(1, CRASH_WINDOW) == 14, "a genuinely new batch must still be applied"

print()
print("✅ Replays are absorbed; new batches still count.")
print("   Note what this does NOT give you: the aggregate is still additive, so")
print("   this only protects against replaying the SAME batch id. A consumer that")
print("   re-partitions its batches differently after a restart produces different")
print("   ids and slips through. That residual risk is what the batch")
print("   reconciliation job in Step 9 exists to catch.")


## 🔄 Step 7 — Full Pipeline: Produce → Aggregate → Query

Let's run the complete pipeline end-to-end: produce a burst of events, aggregate them, flush, and query.

In [ ]:
# Clean slate for this demo
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM click_aggregates")
conn.commit()
conn.close()

# ── PRODUCE ──
pipeline_topic = "ad-clicks-pipeline"
admin = AdminClient({"bootstrap.servers": KAFKA_BROKER})
existing = admin.list_topics(timeout=10).topics
if pipeline_topic not in existing:
    futures = admin.create_topics([NewTopic(pipeline_topic, num_partitions=3, replication_factor=1)])
    for t, f in futures.items():
        f.result()

producer = Producer({"bootstrap.servers": KAFKA_BROKER})

base = datetime(2026, 3, 31, 14, 0, 0, tzinfo=timezone.utc)
NUM_ADS, NUM_MINUTES = 5, 8

# Give every (ad, minute) pair a non-zero number of clicks. A dense series is
# what the sliding-window section in Step 8 needs as its starting point — it
# then punches a hole in it on purpose to show what a hole does.
planned = {
    (ad_id, minute): random.randint(3, 15)
    for ad_id in range(1, NUM_ADS + 1)
    for minute in range(NUM_MINUTES)
}
num_events = sum(planned.values())

for (ad_id, minute), count in planned.items():
    for _ in range(count):
        event_time = base + timedelta(minutes=minute, seconds=random.randint(0, 59))
        event = {
            "ad_id": ad_id,
            "impression_id": str(uuid.uuid4()),
            "user_id": f"user_{random.randint(1, 100)}",
            "ip_address": f"10.0.1.{random.randint(1,254)}",
            "user_agent": "Mozilla/5.0",
            "event_time": event_time.isoformat(),
            "run_id": RUN_ID,
        }
        producer.produce(pipeline_topic, key=str(ad_id), value=json.dumps(event))

producer.flush(timeout=10)
print(f"📤 Produced {num_events} events across {NUM_MINUTES} minutes for {NUM_ADS} ads")

# ── AGGREGATE ──
consumed, windows = aggregate_clicks(pipeline_topic, max_events=num_events,
                                     timeout_s=30, run_id=RUN_ID)
print(f"📥 Consumed {consumed} events, aggregated into {len(windows)} windows")

assert consumed == num_events, \
    f"produced {num_events} events but only consumed {consumed} — the pipeline lost clicks"
assert len(windows) == NUM_ADS * NUM_MINUTES, \
    f"expected {NUM_ADS * NUM_MINUTES} (ad, minute) windows, got {len(windows)}"

# ── FLUSH ──
flushed = flush_to_postgres(windows)
print(f"💾 Flushed {flushed} aggregates to PostgreSQL")

# ── QUERY ──
conn = get_db()
cur = conn.cursor()

# Advertiser dashboard: total clicks per ad across all windows
start = time.time()
cur.execute("""
    SELECT a.title, SUM(ca.click_count) AS total_clicks,
           SUM(ca.unique_users) AS est_unique
    FROM click_aggregates ca
    JOIN ads a ON a.id = ca.ad_id
    GROUP BY a.id, a.title
    ORDER BY total_clicks DESC
""")
rows = cur.fetchall()
elapsed = (time.time() - start) * 1000
conn.close()

print(f"\n⚡ Advertiser dashboard query: {elapsed:.1f} ms\n")
print(f"{'Ad Title':<42} {'Total Clicks':>13} {'~Unique Users':>14}")
print("-" * 72)
for row in rows:
    print(f"{row[0]:<42} {row[1]:>13} {row[2]:>14}")

total_stored = sum(row[1] for row in rows)
assert total_stored == num_events, \
    f"pre-aggregates hold {total_stored} clicks but {num_events} were produced"

print("\n✅ Full pipeline: Produce → Aggregate → Query — all in seconds,")
print(f"   and every one of the {num_events} clicks is accounted for.")
print("   (The ~Unique Users column SUMs per-window unique counts, so a user who")
print("    clicked in two minutes is counted twice — it is an upper bound, not a")
print("    distinct count. See the note in Step 3.)")

## 🌊 Step 8 — Sliding Windows

Tumbling windows are great for "how many clicks in minute 10:00?" But advertisers often want **rolling metrics** like "how many clicks in the *last 5 minutes*, updated every minute?" That's a **sliding window**.

```
Time ──────────────────────────────────────────────▶

Tumbling (1-min, non-overlapping):
│ 10:00 │ 10:01 │ 10:02 │ 10:03 │ 10:04 │

Sliding (5-min window, 1-min step — overlapping!):
│───── 10:00-10:05 ─────│
        │───── 10:01-10:06 ─────│
                │───── 10:02-10:07 ─────│
```

A click at 10:02 contributes to **five** sliding windows (10:02–10:07, 10:01–10:06, 10:00–10:05, …). That sounds expensive — but there's a beautiful trick.

### The Trick: Sum Tumbling Windows

We already have 1-minute tumbling aggregates in `click_aggregates`. A 5-minute sliding total is just **the sum of the tumbling buckets covering the last 5 minutes**. We don't need to re-process raw events — we compose sliding answers from the pre-aggregated data.

This is exactly what real systems (Flink, Druid, Pinot) do under the hood.

### The trap: rows are not minutes

The obvious SQL is `ROWS BETWEEN 4 PRECEDING AND CURRENT ROW`. It is wrong, and it
is wrong in a way that never throws an error.

A minute in which an ad got **zero** clicks produces **no row**. `ROWS` counts
rows, so on a series with a quiet minute in it, "the previous 4 rows" silently
reaches back **six** minutes and reports a 5-minute number that is too big.

The fix is `RANGE BETWEEN INTERVAL '4 minutes' PRECEDING AND CURRENT ROW`, which
measures the frame in **event time** and treats a missing minute as the zero it
actually is. Below we run both, side by side, on a series with a deliberate hole.


In [ ]:
# Compute 5-minute sliding totals by summing the 1-minute tumbling rows that
# the full pipeline (Step 7) just wrote.

conn = get_db()
cur = conn.cursor()

# Ad 1 has a quiet minute: no clicks, so no row. This happens constantly in real
# traffic; here we create it on purpose so the two window frames can disagree.
cur.execute("""
    DELETE FROM click_aggregates
    WHERE ad_id = 1
      AND window_start = (
          SELECT MIN(window_start) + INTERVAL '2 minutes'
          FROM click_aggregates WHERE ad_id = 1
      )
""")
gap_rows = cur.rowcount
conn.commit()
print(f"🕳️  Ad 1 had a quiet minute — removed {gap_rows} row to create the gap\n")

cur.execute("""
    SELECT
        a.title,
        ca.window_start,
        ca.click_count AS clicks_in_minute,
        -- WRONG: counts the previous four ROWS, whatever timestamps they carry.
        SUM(ca.click_count) OVER (
            PARTITION BY ca.ad_id
            ORDER BY ca.window_start
            ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
        ) AS rolling_by_rows,
        -- RIGHT: counts the previous four MINUTES of event time, gaps included.
        SUM(ca.click_count) OVER (
            PARTITION BY ca.ad_id
            ORDER BY ca.window_start
            RANGE BETWEEN INTERVAL '4 minutes' PRECEDING AND CURRENT ROW
        ) AS rolling_5min_total
    FROM click_aggregates ca
    JOIN ads a ON a.id = ca.ad_id
    ORDER BY a.title, ca.window_start
""")
rows = cur.fetchall()
conn.close()

print("🌊 5-Minute Sliding Window (step = 1 minute)")
print("=" * 92)
print(f"{'Ad':<34} {'Window Start':<18} {'This Min':>8} {'ROWS':>7} {'RANGE':>7}  ")
print("-" * 92)
for title, window_start, minute_clicks, by_rows, by_range in rows:
    flag = "  ← ROWS reached too far back" if by_rows != by_range else ""
    print(f"{title[:33]:<34} {window_start.strftime('%Y-%m-%d %H:%M'):<18} "
          f"{minute_clicks:>8} {by_rows:>7} {by_range:>7}{flag}")

disagreements = [r for r in rows if r[3] != r[4]]

print()
print("💡 The RANGE column is the answer you want: the sum of the last 5 minutes of")
print("   event time, composed from pre-aggregates with no raw-event scan.")
print(f"   The ROWS column is wrong on {len(disagreements)} row(s) — every one of them")
print("   downstream of ad 1's quiet minute, where 'the last 5 rows' spans 6 minutes.")
print()
print("   Note also that the first four rows of every ad are PARTIAL windows: there")
print("   simply is not 5 minutes of history behind them yet. Dashboards either")
print("   suppress those or label them, they do not present them as 5-minute totals.")

# ── Assertions ─────────────────────────────────────────────────────────────
assert gap_rows == 1, "the demo needs exactly one deleted minute to open a gap"
assert disagreements, (
    "the ROWS-vs-RANGE trap must reproduce — with a gap in the series and more "
    "than 5 minutes of data, the two frames have to disagree somewhere"
)
assert all(by_rows >= by_range for _, _, _, by_rows, by_range in rows), (
    "ROWS can only ever reach further back than RANGE, never less far — "
    "if this fires the two frames are not what we think they are"
)


### Sliding Windows in Pure Python (No SQL)

Want to see it without SQL? Here's the same logic in pure Python — useful if you're computing sliding totals in a stream processor like Flink or Kafka Streams.


In [ ]:
from collections import deque

def sliding_totals(per_minute_counts, window_minutes: int = 5):
    """Given a list of per-minute click counts, return the rolling window_minutes
    total for each minute. Uses a deque for O(1) amortized updates.

    The input must be DENSE — one entry per minute, zeros included. Hand this a
    list with the quiet minutes squeezed out and you reintroduce exactly the
    ROWS-vs-RANGE bug from the SQL above, because position in the list is the
    only clock this function has.
    """
    window = deque()
    running_sum = 0
    result = []
    for count in per_minute_counts:
        window.append(count)
        running_sum += count
        if len(window) > window_minutes:
            running_sum -= window.popleft()
        result.append(running_sum)
    return result

# Pretend ad 1 had these click counts per minute for 10 minutes
per_minute = [5, 8, 12, 30, 45, 60, 22, 18, 10, 7]  # spike around minute 5!
rolling = sliding_totals(per_minute, window_minutes=5)

print("🌊 Pure-Python Sliding Totals (5-minute window)")
print("=" * 55)
print(f"{'Minute':<8} {'This Minute':<14} {'Rolling 5m':<12}")
print("-" * 55)
for i, (m, r) in enumerate(zip(per_minute, rolling)):
    bar = "█" * (r // 10)
    print(f"{i:<8} {m:<14} {r:<12} {bar}")

print()
print("💡 The rolling total smooths out spikes and is what powers dashboards like:")
print("   'clicks in the last 5 minutes', 'CTR over the last hour', etc.")

# ── Assertions: the O(1) deque must agree with the O(n) definition ─────────
brute_force = [sum(per_minute[max(0, i - 4):i + 1]) for i in range(len(per_minute))]
assert rolling == brute_force, \
    f"deque and brute force disagree:\n  deque {rolling}\n  brute {brute_force}"
# The first 4 entries are partial windows — fewer than 5 minutes exist behind them.
assert rolling[:5] == [5, 13, 25, 55, 100], f"leading partial windows wrong: {rolling[:5]}"
assert rolling[4] == sum(per_minute[:5])
assert max(rolling) == sum(per_minute[3:8]), "the 5-minute peak should cover the spike"
print("✅ deque matches brute force; the first 4 values are partial windows.")


## 🧮 Step 9 — Reconciliation & Lambda Architecture

Streaming is fast but **not always perfectly accurate**:

- A consumer might crash mid-batch and re-process some events (double-counting).
- Late-arriving events might be dropped if they miss the watermark.
- A bug in the aggregation code could silently drift over time.

The fix is the **Lambda Architecture**: run a second, slower, highly-accurate batch pipeline alongside the streaming one. Periodically **reconcile** the two. Whenever they disagree, the batch result wins and corrects the streaming aggregates.

```
Raw events (source of truth)
     │
     ├──▶  🚀 Streaming path   ──▶  click_aggregates  (fast, approximate)
     │                                    ▲
     │                                    │ overwrite on mismatch
     └──▶  🧮 Batch job (hourly) ─────────┘    (slow, exact)
```

Drift comes in **two shapes**, and a reconciliation job that only looks for one of
them is worse than useless, because it reports "all clean":

| Shape | Cause | What the aggregates look like |
|---|---|---|
| **Under-count** | consumer crashed mid-flush, watermark dropped late events | a window exists, its count is too low |
| **Phantom / over-count** | a replayed batch, a bad backfill | a window exists in `click_aggregates` with **no raw events behind it at all** |

A phantom row is invisible to a `LEFT JOIN` starting from the raw events — there
is no truth row to join *from*. Catching it needs a `FULL OUTER JOIN` and a
`DELETE` for the orphans. We inject one of each and check the job finds both.


In [ ]:
# First, produce a fresh batch where raw events AND aggregates both exist.
reconcile_topic = f"ad-clicks-reconcile-{uuid.uuid4().hex[:8]}"  # unique per run for clean demos
admin = AdminClient({"bootstrap.servers": KAFKA_BROKER})
if reconcile_topic not in admin.list_topics(timeout=10).topics:
    for t, f in admin.create_topics(
        [NewTopic(reconcile_topic, num_partitions=3, replication_factor=1)]
    ).items():
        f.result()

producer = Producer({"bootstrap.servers": KAFKA_BROKER})
base = datetime(2026, 3, 31, 11, 0, 0, tzinfo=timezone.utc)

# Clear and write a known set of raw events + aggregates
conn = get_db(); cur = conn.cursor()
cur.execute("DELETE FROM click_aggregates")
cur.execute("DELETE FROM click_events")
conn.commit()

raw_events = []
for minute in range(3):
    for ad_id in (1, 2):
        count = random.randint(5, 15)
        for _ in range(count):
            event_time = base + timedelta(minutes=minute, seconds=random.randint(0, 59))
            evt = {
                "ad_id": ad_id,
                "impression_id": str(uuid.uuid4()),
                "user_id": f"user_{random.randint(1,50)}",
                "ip_address": "10.0.0.1",
                "user_agent": "Mozilla/5.0",
                "event_time": event_time.isoformat(),
                "run_id": RUN_ID,
            }
            raw_events.append(evt)
            # Write the raw event to Postgres (source of truth)
            cur.execute(
                "INSERT INTO click_events (ad_id, impression_id, user_id, "
                "ip_address, user_agent, event_time) VALUES (%s,%s,%s,%s,%s,%s)",
                (evt["ad_id"], evt["impression_id"], evt["user_id"],
                 evt["ip_address"], evt["user_agent"], evt["event_time"]),
            )
            # And publish to Kafka for the streaming path
            producer.produce(reconcile_topic, key=str(ad_id), value=json.dumps(evt))

conn.commit(); conn.close()
producer.flush(timeout=10)

# Run the streaming aggregation (reuses aggregate_clicks + flush_to_postgres from Step 2/3)
_consumed, windows = aggregate_clicks(reconcile_topic, max_events=len(raw_events),
                                      timeout_s=30, run_id=RUN_ID)
flush_to_postgres(windows)
print(f"✅ Produced {len(raw_events)} raw events and ran streaming aggregation.")

assert _consumed == len(raw_events), (
    f"streaming path consumed {_consumed} of {len(raw_events)} events — the drift "
    "below has to be the drift we inject, not one we caused by accident"
)


In [ ]:
# Two kinds of streaming failure, injected on purpose:
#   1. an UNDER-count — the consumer crashed mid-flush and lost 3 clicks
#   2. a PHANTOM window — an aggregate row with no raw events behind it at all,
#      which is the shape a replayed batch or a bad backfill leaves behind

conn = get_db(); cur = conn.cursor()
cur.execute("""
    UPDATE click_aggregates
    SET click_count = click_count - 3
    WHERE window_start = '2026-03-31 11:00:00' AND ad_id = 1
""")
drifted_rows = cur.rowcount
cur.execute("""
    INSERT INTO click_aggregates (ad_id, window_start, click_count, unique_users)
    VALUES (2, '2026-03-31 11:59:00', 42, 7)
""")
conn.commit()
print(f"💥 Injected drift:   ad_id=1 @ 11:00 is 3 clicks short ({drifted_rows} row)")
print("💥 Injected phantom: ad_id=2 @ 11:59 claims 42 clicks that never happened")
print()

# ── The RECONCILIATION BATCH JOB ────────────────────────────────────────────
# `click_events` is the source of truth. Note there is no `AT TIME ZONE` here:
# both event_time and window_start are TIMESTAMP WITHOUT TIME ZONE holding UTC,
# so date_trunc alone lands on exactly the same value the streaming path wrote.
# Converting through timestamptz would silently shift every window by the
# server's TimeZone setting.
#
# And note the FULL OUTER JOIN. A LEFT JOIN from `truth` only sees windows that
# have raw events, so the phantom row above would never show up.
RECONCILE_SQL = """
    WITH truth AS (
        SELECT
            ad_id,
            date_trunc('minute', event_time) AS window_start,
            COUNT(*)                 AS true_count,
            COUNT(DISTINCT user_id)  AS true_unique
        FROM click_events
        GROUP BY ad_id, date_trunc('minute', event_time)
    )
    SELECT COALESCE(t.ad_id, ca.ad_id)               AS ad_id,
           COALESCE(t.window_start, ca.window_start) AS window_start,
           COALESCE(t.true_count, 0)                 AS true_count,
           COALESCE(ca.click_count, 0)               AS streaming_count
    FROM truth t
    FULL OUTER JOIN click_aggregates ca
      ON ca.ad_id = t.ad_id AND ca.window_start = t.window_start
    WHERE COALESCE(t.true_count, 0) <> COALESCE(ca.click_count, 0)
    ORDER BY 1, 2
"""

cur.execute(RECONCILE_SQL)
mismatches = cur.fetchall()
print(f"🔍 Reconciliation found {len(mismatches)} mismatch(es):")
for ad_id, ws, truth, streaming in mismatches:
    kind = "phantom" if truth == 0 else "under-count"
    print(f"   ad={ad_id} window={ws} truth={truth} streaming={streaming} "
          f"(Δ={truth - streaming:+d}, {kind})")

# ── Repair: batch wins ──────────────────────────────────────────────────────
# Overwrite (not add to) every window that has raw events...
cur.execute("""
    INSERT INTO click_aggregates (ad_id, window_start, click_count, unique_users)
    SELECT
        ad_id,
        date_trunc('minute', event_time),
        COUNT(*),
        COUNT(DISTINCT user_id)
    FROM click_events
    GROUP BY ad_id, date_trunc('minute', event_time)
    ON CONFLICT (ad_id, window_start) DO UPDATE
    SET click_count  = EXCLUDED.click_count,
        unique_users = EXCLUDED.unique_users,
        updated_at   = CURRENT_TIMESTAMP
""")
repaired = cur.rowcount

# ...and delete every window that has none. Without this the phantom survives
# the "repair" untouched, and the job reports success while over-billing.
cur.execute("""
    DELETE FROM click_aggregates ca
    WHERE NOT EXISTS (
        SELECT 1 FROM click_events ce
        WHERE ce.ad_id = ca.ad_id
          AND date_trunc('minute', ce.event_time) = ca.window_start
    )
""")
orphans = cur.rowcount
conn.commit()

# Run the same comparison again — a reconciliation job that cannot verify itself
# is just a second opinion.
cur.execute(RECONCILE_SQL)
remaining = cur.fetchall()
conn.close()

print()
print(f"🛠️  Repaired/refreshed {repaired} aggregate rows from raw events.")
print(f"🗑️  Deleted {orphans} phantom row(s) with no raw events behind them.")
print(f"🔁 Mismatches remaining after repair: {len(remaining)}")

# ── Assertions ─────────────────────────────────────────────────────────────
assert drifted_rows == 1, "the drift injection must hit exactly one row"
assert len(mismatches) == 2, (
    "reconciliation must catch BOTH the under-count and the phantom; "
    f"it found {len(mismatches)}: {mismatches}"
)
assert any(t == 0 for _, _, t, _ in mismatches), \
    "the phantom row (truth=0) must be one of the mismatches — check the FULL OUTER JOIN"
assert orphans == 1, f"exactly one phantom row should be deleted, deleted {orphans}"
assert remaining == [], f"reconciliation left {len(remaining)} mismatch(es) behind: {remaining}"

print()
print("💡 In production this job runs hourly or daily. Advertisers see fast")
print("   streaming numbers immediately; a few hours later those numbers become")
print("   exact via reconciliation. That's the Lambda Architecture in action.")
print()
print("   What this toy version skips: the real job is incremental (it only")
print("   re-reads the last few hours, not the whole table), it writes a corrected")
print("   copy rather than mutating live rows advertisers are reading, and it")
print("   alerts when the drift exceeds a threshold instead of silently fixing it.")


## 🧹 Cleanup

In [ ]:
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM click_aggregates")
cur.execute("DELETE FROM click_events")
cur.execute("DROP TABLE IF EXISTS applied_batches")   # created in Step 6
conn.commit()
conn.close()
print("🧹 Cleaned up PostgreSQL tables")
print("   (Kafka topics still exist — that's fine for Notebook 3)")

## 📚 Summary

### Key Takeaways

1. **Pre-aggregation** trades storage for query speed — one row per ad per minute instead of millions of raw events
2. **Event time > processing time** — always bucket clicks by when they *happened*, not when you *received* them
3. **Tumbling windows** are half-open `[start, end)`, so a click landing exactly on a boundary
   belongs to the window that *opens* there — exactly one window, never two, never none
4. **Late events** are absorbed by the UPSERT *while the window is still open*; a **watermark**
   (`max event time − allowed lateness`) is what decides when it stops being open. Everything
   past that is dropped, and dropped clicks are silent under-counting
5. **At-least-once delivery double-counts.** An additive UPSERT applied twice bills the
   advertiser twice. An **idempotency key** — a deterministic batch id written in the same
   transaction as the counts — is what makes the flush safe to retry
6. **Sliding windows compose out of tumbling ones**, but use `RANGE ... INTERVAL`, not `ROWS`:
   a minute with zero clicks writes no row, and `ROWS` cannot tell the difference
7. **Reconciliation must look both ways.** Under-counts *and* phantom rows; a `LEFT JOIN` from
   the raw events finds only the first kind
8. **Flink** does most of this for you in production — watermarks, state management, and an
   exactly-once sink — which is precisely why it is worth knowing what it is doing

### What's Next

In **Notebook 3**, we tackle the *other* source of duplicate clicks — the client side.
Step 6 above fixed replays inside our own pipeline; Notebook 3 stops the same click
arriving twice in the first place, with impression IDs, HMAC signing and a Redis
dedup cache.